# Tabla de hechos: fact_suscripciones_cuentas

## Objetivo
Esta tabla de hechos calcula **métricas y campos útiles** para el análisis de suscripciones:
* Duración de cada suscripción en días
* Tipo de cambio (nueva suscripción, upgrade, downgrade)
* Identificación de churn y métricas temporales asociadas
* Días desde el cambio anterior y desde el inicio de la cuenta

## Diseño técnico

### Joins con dimensiones
Se realizan **joins con las tablas de dimensiones** (`dim_cuentas`, `dim_suscripciones`) para recuperar las **claves subrogadas** (surrogate keys) correspondientes a cada fecha de inicio de suscripción, respetando la validez temporal (`valido_desde` y `valido_hasta`).

### Estrategia de carga: INSERT OVERWRITE
Se utiliza **INSERT OVERWRITE** en lugar de carga incremental por las siguientes razones:
* La tabla es **relativamente pequeña**, lo que hace viable la recarga completa
* Garantiza **idempotencia**: ejecutar múltiples veces produce el mismo resultado
* **Evita problemas con claves subrogadas autoincrementales**: si se recarga alguna dimensión y cambian los IDs subrogados, una carga incremental podría:
  * Romper las referencias existentes
  * Apuntar a registros incorrectos en las dimensiones
  * Generar inconsistencias en los análisis

> **Nota**: Si la tabla creciera significativamente, se debería evaluar una estrategia de carga incremental con mecanismos de reconciliación de claves subrogadas.

In [0]:
-- Cargar tabla: fact_suscripciones_cuentas con parámetros dinámicos

INSERT OVERWRITE IDENTIFIER(:catalogo_destino || '.analisis_suscripciones_' || :alumno || '.fact_suscripciones_cuentas')

WITH suscripciones_ordenadas AS (
    SELECT
        id_suscripcion_cuenta,
        id_cuenta,
        id_suscripcion,
        fecha_inicio,
        fecha_fin,
        -- Obtener la suscripción anterior para comparar
        LAG(id_suscripcion) OVER (
            PARTITION BY id_cuenta
            ORDER BY fecha_inicio
        ) AS id_suscripcion_anterior,
        -- Obtener la fecha de inicio de la suscripción anterior
        LAG(fecha_inicio) OVER (
            PARTITION BY id_cuenta
            ORDER BY fecha_inicio
        ) AS fecha_inicio_anterior,
        -- Obtener la primera suscripción de la cuenta
        FIRST_VALUE(fecha_inicio) OVER (
            PARTITION BY id_cuenta
            ORDER BY fecha_inicio
        ) AS fecha_primer_suscripcion,
        -- Obtener la siguiente suscripción para saber si es churn
        LEAD(id_suscripcion) OVER (
            PARTITION BY id_cuenta
            ORDER BY fecha_inicio
        ) AS id_suscripcion_siguiente
    FROM IDENTIFIER(:catalogo_origen || '.core_negocio_' || :alumno || '.cuentas_suscripciones')
),

suscripciones_calculadas AS (
    SELECT
        *,
        -- Calcular duración de la suscripción en días
        DATEDIFF(DAY, fecha_inicio, COALESCE(fecha_fin, CURRENT_DATE())) AS duracion_dias,
        -- Determinar el tipo de cambio
        CASE
            WHEN id_suscripcion_anterior IS NULL THEN 'Nueva Suscripción'
            WHEN id_suscripcion > id_suscripcion_anterior THEN 'Upgrade'
            WHEN id_suscripcion < id_suscripcion_anterior THEN 'Downgrade'
            ELSE 'Misma Suscripción'
        END AS tipo_cambio,
        -- Fecha de churn (solo cuando no hay siguiente suscripción)
        CASE 
            WHEN fecha_fin IS NOT NULL AND id_suscripcion_siguiente IS NULL 
            THEN fecha_fin 
            ELSE NULL 
        END AS fecha_churn,
        -- Días desde el cambio anterior hasta el churn
        CASE 
            WHEN fecha_fin IS NOT NULL AND fecha_inicio_anterior IS NOT NULL 
            AND id_suscripcion_siguiente IS NULL
            THEN DATEDIFF(DAY, fecha_inicio_anterior, fecha_fin)
            ELSE NULL
        END AS dias_desde_cambio_anterior,
        -- Días desde la primera suscripción hasta el churn
        CASE 
            WHEN fecha_fin IS NOT NULL AND id_suscripcion_siguiente IS NULL
            THEN DATEDIFF(DAY, fecha_primer_suscripcion, fecha_fin)
            ELSE NULL
        END AS dias_desde_inicio_cuenta
    FROM suscripciones_ordenadas
),

dimensiones AS (
    SELECT
        -- Buscar ID de dimensión de cuenta válido para la fecha de inicio
        c.id_dim_cuenta,
        -- Buscar ID de dimensión de suscripción válido para la fecha de inicio
        sub.id_dim_suscripcion,
        -- Buscar ID de dimensión de suscripción anterior válido para la fecha de inicio
        sub_ant.id_dim_suscripcion AS id_dim_suscripcion_anterior,
        s.fecha_inicio,
        s.fecha_fin,
        s.duracion_dias,
        s.tipo_cambio,
        s.fecha_churn,
        s.dias_desde_cambio_anterior,
        s.dias_desde_inicio_cuenta
    FROM suscripciones_calculadas s
    LEFT JOIN IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_cuentas') c
        ON s.id_cuenta = c.id_cuenta
        AND s.fecha_inicio BETWEEN c.valido_desde AND c.valido_hasta
    LEFT JOIN IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_suscripciones') sub
        ON s.id_suscripcion = sub.id_suscripcion
        AND s.fecha_inicio BETWEEN sub.valido_desde AND sub.valido_hasta
    LEFT JOIN IDENTIFIER(:catalogo_destino || '.dimensiones_comunes_' || :alumno || '.dim_suscripciones') sub_ant
        ON s.id_suscripcion_anterior = sub_ant.id_suscripcion
        AND s.fecha_inicio BETWEEN sub_ant.valido_desde AND sub_ant.valido_hasta
)

SELECT * FROM dimensiones

In [0]:
select * from IDENTIFIER(:catalogo_destino || '.analisis_suscripciones_' || :alumno || '.fact_suscripciones_cuentas') limit 10